In [1]:
from fuzzycar import Car
from pynq import Overlay , MMIO

In [2]:
ol = Overlay("car.bit", ignore_version=True)
print("✔ Overlay 'car.bit' loaded successfully.\n")

✔ Overlay 'car.bit' loaded successfully.



In [3]:
car = Car(ol)

Configuring SPI controller...


In [4]:
gpio = MMIO(ol.axi_gpio_0.mmio.base_addr, 0x10000 , debug=False)
gpio.write(0x00,0xff)

In [5]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import time
from IPython.display import clear_output
print("Fuzzy imported libraries complete!")

Fuzzy imported libraries complete!


In [6]:
# Define fuzzy variables
Error=ctrl.Antecedent(np.arange(-0.9,1.01,0.01),'Error')
Delta_Error=ctrl.Antecedent(np.arange(-1.9,1.9,0.01),'Delta_Error')
print("Fuzzy variables defined")

Fuzzy variables defined


In [7]:
# Membership functions

# Membership function for Error
Error['Close']=fuzz.trapmf(Error.universe,[-0.9, -0.9, -0.15, -0.05])
Error['None']=fuzz.trimf(Error.universe,[-0.2, 0, 0.2])
Error['Far']=fuzz.trapmf(Error.universe,[0.05, 0.15, 1, 1])
print("Error MFs defined")

# Membership function for Delta_Error
Delta_Error['movingCloser']=fuzz.trapmf(Delta_Error.universe,[-1.9, -1.9, -0.15, -0.05])
Delta_Error['Stable']=fuzz.trimf(Delta_Error.universe,[-0.2, 0, 0.2])
Delta_Error['movingFarther']=fuzz.trapmf(Delta_Error.universe,[0.05, 0.15, 1.9, 1.9])
print("Delta_Error MFs defined")

Error MFs defined
Delta_Error MFs defined


In [8]:
outputs = {
    'RevFast': 1440,
    'RevMed':  1450,
    'RevSlow': 1460,
    'Idle':    1600,
    'ForwSlow':1669,
    'ForwMed': 1675,
    'ForwFast':1690
}
print("Sugeno output constants defined")

Sugeno output constants defined


In [9]:
# Rule base
rules = [
    ('Close', 'movingCloser',  'RevSlow'),
    ('Close', 'Stable',        'RevMed'),
    ('Close', 'movingFarther', 'RevFast'),

    ('None',  'movingCloser',  'ForwSlow'),
    ('None',  'Stable',        'Idle'),
    ('None',  'movingFarther', 'RevSlow'),

    ('Far',   'movingCloser',  'ForwSlow'),
    ('Far',   'Stable',        'ForwMed'),
    ('Far',   'movingFarther', 'ForwFast')
]
print("Fuzzy rules defined")

Fuzzy rules defined


In [10]:
# Fuzzify inputs

def get_memberships(error, delta_error):
    mu_error = {
        'Close': fuzz.interp_membership(Error.universe, Error['Close'].mf, error),
        'None':  fuzz.interp_membership(Error.universe, Error['None'].mf, error),
        'Far':   fuzz.interp_membership(Error.universe, Error['Far'].mf, error)
    }

    mu_delta = {
        'movingCloser':  fuzz.interp_membership(Delta_Error.universe, Delta_Error['movingCloser'].mf, delta_error),
        'Stable':        fuzz.interp_membership(Delta_Error.universe, Delta_Error['Stable'].mf, delta_error),
        'movingFarther': fuzz.interp_membership(Delta_Error.universe, Delta_Error['movingFarther'].mf, delta_error)
    }

    return mu_error, mu_delta

In [11]:
# Sugeno inference

def compute_speed(error_value, delta_error_value):
    mu_error, mu_delta = get_memberships(error_value, delta_error_value)

    numerator = 0.0
    denominator = 0.0

    for error_label, delta_label, output_label in rules:
        firing_strength = min(mu_error[error_label], mu_delta[delta_label])   # AND = min
        output_value = outputs[output_label]

        numerator += firing_strength * output_value
        denominator += firing_strength

    if denominator == 0:
        return outputs['Idle']

    return numerator / denominator

print("Sugeno controller function defined")

Sugeno controller function defined


In [ ]:
# For testing controller only//no need to run!!!

error_value = 0.2
delta_error_value = 1.8

speed_output = compute_speed(error_value, delta_error_value)
speed_pwm = int(round(speed_output))

print("Error =", error_value)
print("Delta_Error =", delta_error_value)
print("Sugeno output =", speed_output)
print("PWM command =", speed_pwm)

In [14]:

# for testing reverse speed //no need to run!!!
    #'RevFast': 1400,
    #'RevMed':  1430,
    #'RevSlow': 1460,
    #'Idle':    1600,
    #'ForwSlow':1669,
    #'ForwMed': 1672,
    #'ForwFast':1675
car.motor.set_pwm_time(50, 1600)
while True:
    car.motor.set_pwm_time(50, 1669)
    time.sleep(0.5)
    car.motor.set_pwm_time(50, 1600)
    time.sleep(0.5)
    car.motor.set_pwm_time(50,1460)
    time.sleep(0.5)
#esc motor preven vehicle from driving throttle, then reversing. This is due to protect the drivetrain
# Pulse for those 3 cases appear in the digital oscilloscope.

KeyboardInterrupt: 

In [12]:
import sys
import subprocess

def check_lib(package_name):
    try:
        dist = __import__(package_name)
        print(f"✅ {package_name} is installed. Version: {getattr(dist, '__version__', 'Unknown')}")
        return True
    except ImportError:
        print(f"❌ {package_name} is MISSING.")
        return False

print("--- Python Library Check ---")
opencv_ok = check_lib("cv2")
numpy_ok = check_lib("numpy")

if opencv_ok:
    import cv2
    # Check specifically for ArUco module in OpenCV
    if hasattr(cv2, 'aruco'):
        print("✅ ArUco module found in cv2.")
    else:
        print("❌ cv2 found, but ArUco module is MISSING (Need opencv-contrib-python).")

print("\n--- System Hardware Support Check ---")
# Check for libatlas (needed for fast math on ARM)
try:
    lib_check = subprocess.check_output("ldconfig -p | grep libatlas", shell=True)
    print("✅ libatlas-base-dev is linked and ready.")
except subprocess.CalledProcessError:
    print("❌ libatlas-base-dev is MISSING (Run 'sudo apt install libatlas-base-dev').")

--- Python Library Check ---
✅ cv2 is installed. Version: 4.5.4
✅ numpy is installed. Version: 1.21.5
✅ ArUco module found in cv2.

--- System Hardware Support Check ---
✅ libatlas-base-dev is linked and ready.


In [ ]:
import sys
import subprocess
from IPython.display import display, Javascript
import math
import os
import threading
import time
from flask import Flask, Response
import cv2
import cv2.aruco as aruco
import numpy as np

# 1. Setup Environment and Legacy ArUco Parameters
os.environ["OPENCV_LOG_LEVEL"] = "ERROR"
app = Flask(__name__)

# LEGACY initialization for OpenCV 4.5.4 (Fixes image_b98640)
aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)
parameters = aruco.DetectorParameters_create()

# 2. Initialize Camera with V4L2 Backend (Fixes image_b985fc)
video_in = cv2.VideoCapture(0, cv2.CAP_V4L2)
# --- RESOLUTION SETTINGS ---
# 640x480 forces a 4:3 ratio. If the image still looks squished after running this, 
# change these to a 16:9 ratio like 640x360 or 1280x720 to match your webcam's native sensor.
video_in.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
video_in.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# Define our dashboard colors and formal font
LIGHT_BLUE = (230, 216, 173) 
GREEN = (0, 255, 0)
FONT = cv2.FONT_HERSHEY_DUPLEX

prev_error=0
#set to neutral
car.motor.set_pwm_time(50, 1600)
lastmode="neutral"
def de_speed_controller(pixel_area: int):
    global prev_error, lastmode
    if(pixel_area>100000):
        pixel_area=100000
    error=(10000-pixel_area)
    if(error>0):
        error=error*0.0001
    else:
        error=error*0.00001
    de_error=error-prev_error
    prev_error=error
    speed_output= compute_speed(error, de_error)
    #if(speed_output<1600):
    #    if(lastmode!="reverse"):
    #        car.motor.set_pwm_time(50, 1600)
    #        time.sleep(0.2)
    #        lastmode="reverse"
    #        print(f"lastmode=({lastmode}) & PulseWidth=({int(speed_output)})", end="\r")
    #        return int(speed_output)
    #    #lastmode = "reverse"
    #    print(f"lastmode=({lastmode}) & PulseWidth=({int(speed_output)})", end="\r")
    #    return int(speed_output)
    #else:
    #    lastmode="forward"
    #    print(f"lastmode=({lastmode}) & PulseWidth=({int(speed_output)})", end="\r")
    
    #
    #
    # not needed, hardware issue
     print(f"PulseWidth=({int(speed_output)})", end="\r")
    return int(speed_output)



def gen_frames():
    
    while True:
        success, frame = video_in.read()
        if not success:
            break
        else:
            frame = cv2.flip(frame, -1)
            # Dynamically grab the ACTUAL size the camera is outputting
            frame_h, frame_w = frame.shape[:2]
            
            # --- CREATE THE DYNAMIC DASHBOARD CANVAS ---
            # Make the canvas exactly 120 pixels taller than the camera frame
            canvas = np.zeros((frame_h + 120, frame_w, 3), dtype=np.uint8)
            
            # Paste the exact, unscaled camera frame into the top
            canvas[0:frame_h, 0:frame_w] = frame
            
            # Calculate the absolute center of the frame dynamically
            frame_center_x, frame_center_y = int(frame_w / 2), int(frame_h / 2)
            
            # --- DRAW FRAME CENTER (LIGHT BLUE) ---
            cv2.drawMarker(canvas, (frame_center_x, frame_center_y), LIGHT_BLUE, 
                           markerType=cv2.MARKER_CROSS, markerSize=10, thickness=1)
            cv2.circle(canvas, (frame_center_x, frame_center_y), 2, LIGHT_BLUE, -1)
            
            # Define exact Y-coordinates for the text rows so they stay in the black bar
            row1_y = frame_h + 40
            row2_y = frame_h + 80
            
            # --- WRITE FRAME STATS (LIGHT BLUE) ---
            cv2.putText(canvas, "Frame", (10, row1_y), FONT, 0.45, LIGHT_BLUE, 1)
            cv2.putText(canvas, f"C: {frame_center_x},{frame_center_y}", (110, row1_y), FONT, 0.45, LIGHT_BLUE, 1)
            cv2.putText(canvas, f"W: {frame_w} H: {frame_h}", (250, row1_y), FONT, 0.45, LIGHT_BLUE, 1)
            cv2.putText(canvas, f"Area: {frame_w * frame_h} px^2", (410, row1_y), FONT, 0.45, LIGHT_BLUE, 1)

            # --- ARUCO DETECTION LOGIC ---
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            corners, ids, rejected = aruco.detectMarkers(gray, aruco_dict, parameters=parameters)

            if ids is not None:
                for i in range(len(ids)):
                    c = corners[i][0]
                    
                    # Draw clean polygon box
                    pts = c.astype(np.int32)
                    cv2.polylines(canvas, [pts], isClosed=True, color=GREEN, thickness=2)
                    
                    top_left, top_right, bottom_right, bottom_left = c[0], c[1], c[2], c[3]
                    
                    # Math Calculations
                    width = math.hypot(top_right[0] - top_left[0], top_right[1] - top_left[1])
                    height = math.hypot(bottom_left[0] - top_left[0], bottom_left[1] - top_left[1])
                    area = width * height
                    #
                    #
                    #
                    # very important to feed controller
                    pulseWidth=de_speed_controller(int(area))
                    car.motor.set_pwm_time(50, pulseWidth)
                    center_x, center_y = int(c[:, 0].mean()), int(c[:, 1].mean())
                    
                    
                    
                    # --- DRAW ARUCO CENTER (GREEN) ---
                    cv2.drawMarker(canvas, (center_x, center_y), GREEN, 
                                   markerType=cv2.MARKER_CROSS, markerSize=15, thickness=2)
                    cv2.circle(canvas, (center_x, center_y), 3, GREEN, -1)
                    
                    # --- WRITE ARUCO STATS (GREEN) ---
                    cv2.putText(canvas, f"ID: {ids[i][0]}", (10, row2_y), FONT, 0.45, GREEN, 1)
                    cv2.putText(canvas, f"C: {center_x},{center_y}", (110, row2_y), FONT, 0.45, GREEN, 1)
                    cv2.putText(canvas, f"W: {int(width)} H: {int(height)}", (250, row2_y), FONT, 0.45, GREEN, 1)
                    cv2.putText(canvas, f"Area: {int(area)} px^2", (410, row2_y), FONT, 0.45, GREEN, 1)
                    
            else:
                car.motor.set_pwm_time(50, 1600)
                cv2.putText(canvas, "Searching for Markers...", (10, row2_y), 
                            FONT, 0.45, (0, 0, 255), 1)

            # --- Prepare the CANVAS for Flask ---
            ret, buffer = cv2.imencode('.jpg', canvas)
            frame_bytes = buffer.tobytes()
            yield (b'--frame\r\n'
                   b'Content-Type: image/jpeg\r\n\r\n' + frame_bytes + b'\r\n')

@app.route('/video_feed')
def video_feed():
    return Response(gen_frames(), mimetype='multipart/x-mixed-replace; boundary=frame')

@app.route('/')
def index():
    return '''
    <html>
        <head>
            <style>
                body { margin: 0; background: #000; display: flex; justify-content: center; align-items: center; height: 100vh; }
                img { width: 640px; height: 600; border: 2px solid #00ff00; image-rendering: pixelated; }
            </style>
        </head>
        <body><img src="/video_feed"></body>
    </html>
    '''

def launch_and_run():
    # Start Flask in background
    threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000, debug=False, threaded=True)).start()
    time.sleep(2)
    # Open the window on YOUR laptop browser (ensure IP matches your board)
    display(Javascript('window.open("http://192.168.2.99:5000", "_blank", "width=660,height=610");'))
    print("ArUco Server Online")

if __name__ == '__main__':
    launch_and_run()
    
    try:
        while True:
            time.sleep(0.1)
    except KeyboardInterrupt:
        print("\nCell interrupted. Stopping logs.")